In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin
from sklearn.tree import DecisionTreeRegressor
from sklearn.utils import check_X_y, check_array
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error


class GradientBoosting:
    """
    Gradient Boosting from scratch.
    Supports regression (squared error) and binary classification (log‑loss).
    """
    def __init__(self,
                 n_estimators=100,
                 learning_rate=0.1,
                 max_depth=3,
                 min_samples_split=2,
                 random_state=None):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.random_state = random_state
        self.trees = []
        self.initial_pred = None
        self.classes_ = None
        self.task = None   # 'regression' or 'classification'

    def _sigmoid(self, x):
        return 1.0 / (1.0 + np.exp(-x))

    def _log_odds_prior(self, y):
        """Compute initial log‑odds for classification."""
        p_pos = np.mean(y == self.classes_[1])
        p_pos = np.clip(p_pos, 1e-15, 1 - 1e-15)
        return 0.5 * np.log(p_pos / (1 - p_pos))

    def fit(self, X, y):
        """Fit the gradient boosting model."""
        X, y = check_X_y(X, y)
        self.trees = []
        n_samples = X.shape[0]

        # Determine task and encode labels if needed
        if len(np.unique(y)) == 2:
            self.task = 'classification'
            self.classes_ = np.unique(y)
            # Encode as 0/1 for probability calculation
            y_enc = np.where(y == self.classes_[1], 1, 0)
            # Initial prediction: log‑odds (prior probability)
            self.initial_pred = self._log_odds_prior(y_enc)
            F = np.full(n_samples, self.initial_pred)   # current ensemble prediction
        else:
            self.task = 'regression'
            self.classes_ = None
            # Initial prediction: mean of y
            self.initial_pred = np.mean(y)
            F = np.full(n_samples, self.initial_pred)
            y_enc = y.copy()   # numeric target

        rng = np.random.RandomState(self.random_state)

        for _ in range(self.n_estimators):
            # ---- 1. Compute residuals / pseudo‑residuals ----
            if self.task == 'classification':
                # Pseudo‑residuals for log‑loss: r = y - p, where p = sigmoid(F)
                p = self._sigmoid(F)
                residuals = y_enc - p
            else:
                # Residuals for squared error: r = y - F
                residuals = y_enc - F

            # ---- 2. Fit a regression tree to residuals ----
            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                random_state=rng.randint(0, 10000)
            )
            tree.fit(X, residuals)

            # ---- 3. Compute leaf values (gamma) ----
            # For squared error, gamma = mean(residual) in each leaf.
            # For log‑loss, we could do a line search, but the default Newton
            # step (mean residual) works well with shrinkage.
            leaf_indices = tree.apply(X)   # leaf ids for each sample
            leaf_values = np.zeros(tree.tree_.node_count)
            for leaf_id in range(tree.tree_.node_count):
                # Check if it's a leaf node (left_child == -1)
                if tree.tree_.children_left[leaf_id] == -1:
                    mask = (leaf_indices == leaf_id)
                    if mask.sum() > 0:
                        # For regression: mean residual
                        # For classification: sum(residual) / sum(p*(1-p)) would be exact,
                        # but we use a simpler approximation: mean residual * learning_rate
                        leaf_values[leaf_id] = np.mean(residuals[mask])
            # Function to map leaf indices to gamma values
            def get_gamma(idx):
                return leaf_values[idx]

            # ---- 4. Update ensemble predictions ----
            # F(x) <- F(x) + learning_rate * h_m(x)
            F += self.learning_rate * tree.predict(X)

            # Store the tree and its leaf values for prediction later
            self.trees.append((tree, leaf_values))

        return self

    def predict_raw(self, X):
        """Return the raw ensemble output (log‑odds for classification, value for regression)."""
        X = check_array(X)
        F = np.full(X.shape[0], self.initial_pred)
        for tree, leaf_values in self.trees:
            # Get leaf indices for X in this tree
            leaf_indices = tree.apply(X)
            # Map leaf indices to stored leaf values
            leaf_preds = leaf_values[leaf_indices]
            F += self.learning_rate * leaf_preds
        return F

    def predict(self, X):
        """Final predictions: class labels (classification) or continuous values (regression)."""
        raw = self.predict_raw(X)
        if self.task == 'classification':
            # Convert log‑odds to probabilities, then to class labels
            prob_pos = self._sigmoid(raw)
            return np.where(prob_pos >= 0.5, self.classes_[1], self.classes_[0])
        else:
            return raw

    def predict_proba(self, X):
        """Class probabilities for binary classification."""
        if self.task != 'classification':
            raise ValueError("predict_proba is only available for classification.")
        raw = self.predict_raw(X)
        prob_pos = self._sigmoid(raw)
        prob_pos = np.clip(prob_pos, 1e-15, 1 - 1e-15)
        return np.column_stack((1 - prob_pos, prob_pos))



# Convenience wrappers for scikit‑learn compatibility
class GradientBoostingRegressor(GradientBoosting, RegressorMixin):
    """Gradient Boosting for regression."""
    pass


class GradientBoostingClassifier(GradientBoosting, ClassifierMixin):
    """Gradient Boosting for binary classification."""
    pass



# Example usage
if __name__ == "__main__":
    # ----- Regression example -----
    print("--- Regression ---")
    X_reg, y_reg = make_regression(n_samples=300, n_features=5, noise=10, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)

    gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
    gbr.fit(X_train, y_train)
    y_pred = gbr.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    print(f"MSE: {mse:.3f}")

    # ----- Binary classification example -----
    print("\n--- Binary Classification ---")
    X_cls, y_cls = make_classification(n_samples=300, n_features=10, n_informative=8,
                                       n_redundant=2, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X_cls, y_cls, test_size=0.3, random_state=42)

    gbc = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
    gbc.fit(X_train, y_train)
    y_pred = gbc.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {acc:.3f}")

    # Probabilities
    proba = gbc.predict_proba(X_test[:5])
    print("Predicted probabilities (first 5):\n", proba)

--- Regression ---
MSE: 543.718

--- Binary Classification ---
Accuracy: 0.889
Predicted probabilities (first 5):
 [[0.36840488 0.63159512]
 [0.56410434 0.43589566]
 [0.4595946  0.5404054 ]
 [0.81398122 0.18601878]
 [0.85248829 0.14751171]]
